<a href="https://colab.research.google.com/github/kenleefk-edu/C3669C-2026-05/blob/main/20260528_Llama_3_2_FineTuning_MetaMathQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Llama 3.2 1B on MetaMathQA with Unsloth
This notebook demonstrates the process of fine-tuning the Llama 3.2 1B model using the Unsloth library.  
Unsloth is optimized for faster training and lower memory usage.

`C3669C_LEEFOOKKIN_4582883W`

### Assignment Sections:
1. Environment Setup
2. Data Preparation
3. Fine-tuning Implementation
4. Evaluation and Analysis
5. Documentation and Report

## 1. Environment Setup
In this section, we set up the development environment with GPU support and install necessary dependencies.

In [ ]:
## a. Set up development environment with GPU support
## b. Install required dependencies
## c. Document configuration steps

# Check GPU availability - Google Colab provides T4, V100, or A100 GPUs.
# We use torch to verify that the GPU is accessible.
import torch
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu_stats.name}, Total Memory: {gpu_stats.total_memory / 1024**3:.2f} GB')
else:
    print('No GPU found. Please change runtime type to GPU.')

# Install Unsloth and its dependencies.
# Unsloth is 2x faster and uses 70% less memory than standard Hugging Face fine-tuning.
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install datasets
print('Dependencies installed successfully.')

## 2. Data Preparation
We will use the `meta-math/MetaMathQA` dataset, which is a large-scale dataset for mathematical reasoning.

In [ ]:
import torch
from datasets import load_dataset

# Install unsloth_zoo if it's missing
!pip install unsloth_zoo
from unsloth import FastLanguageModel

# --- Configuration ---
max_seq_length = 2048 # Maximum sequence length supported by the model
dtype = None          # Auto-detection (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True   # 4-bit quantization significantly reduces VRAM usage (essential for Colab free tier)

# a. Load and preprocess dataset
# MetaMathQA contains ~395k rows. We load a subset (10,000 rows) for faster training in this experiment.
dataset = load_dataset('meta-math/MetaMathQA', split='train[:10000]')

# b. Implement data cleaning
# We filter out any rows that might have missing queries or responses.
dataset = dataset.filter(lambda x: x['query'] is not None and x['response'] is not None)

# c. Create training/validation splits
# We split the data: 90% for training and 10% for validation to monitor performance.
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset['train']
val_dataset = dataset['test']

# d. Format data appropriately for the LLM model
# We use the Alpaca-style prompt template to structure the math queries.
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = '<|end_of_text|>' # Standard Llama 3 EOS token to prevent infinite generation

def formatting_prompts_func(examples):
    queries = examples['query']
    responses = examples['response']
    texts = []
    for query, response in zip(queries, responses):
        # Format the query and response into the template and add the EOS token
        text = alpaca_prompt.format(query, response) + EOS_TOKEN
        texts.append(text)
    return { 'text' : texts }

# Apply formatting to both splits
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print(f'Training set size: {len(train_dataset)}')
print(f'Validation set size: {len(val_dataset)}')

### 2e. Justification for dataset choice
**For completeness and as a self-reminder** :  The `MetaMathQA` dataset was chosen because it provides high-quality mathematical reasoning pairs. Fine-tuning on this dataset helps the model learn step-by-step problem-solving logic, which is a critical capability for small models like Llama 3.2 1B. It covers various math levels and rephrased questions, ensuring robustness.

## 3. Fine-tuning Implementation
We load the model and configure LoRA (Low-Rank Adaptation) for efficient parameter updates.

In [ ]:
# --- Load Model ---
# We use the pre-quantized 4-bit version of Llama 3.2 1B from Unsloth.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-1B-bnb-4bit', # The name or path of the pre-trained model to load. No default, must be specified.
    max_seq_length = max_seq_length,              # Maximum sequence length the model can handle. Default varies by model, often 2048 or 4096.
    dtype = dtype,                                # Data type for model weights (e.g., torch.float16, torch.bfloat16). 'None' auto-detects based on GPU. Default is None.
    load_in_4bit = load_in_4bit,                  # Whether to load the model in 4-bit quantization, saving VRAM. Default is False.
)

# --- Configure LoRA Adapters ---
# LoRA allows us to train only a small fraction (1-10%) of the model parameters.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA attention dimension (rank): Controls the number of trainable parameters in LoRA. Higher values allow more complex updates but use more VRAM. Default is 64.
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # List of module names to apply LoRA to. Default depends on the model architecture.
    lora_alpha = 16,      # LoRA scaling factor: Multiplies the LoRA weights. Default is 16.
    lora_dropout = 0,     # The dropout probability for LoRA layers. Optimized to 0 for Unsloth's performance. Default is 0.05.
    bias = 'none',        # Whether to train the bias terms in LoRA layers. Optimized to 'none' for Unsloth. Default is 'none'.
    use_gradient_checkpointing = 'unsloth', # Reduces VRAM usage by recomputing activations during backpropagation. 'unsloth' uses an optimized version. Default is True.
    random_state = 3407,  # Random seed for reproducibility of LoRA layer initialization. Default is None.
    use_rslora = False,   # Whether to use Rank Stabilized LoRA. Default is False.
    loftq_config = None,  # Configuration for LoftQ initialization. Default is None.
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

# a. Configure hyperparameters
training_args = TrainingArguments(
    per_device_train_batch_size = 2,  # Batch size per GPU: This sets the number of training examples processed per device in a single forward/backward pass. Default is 8.
    gradient_accumulation_steps = 8,  # Accumulate gradients: This accumulates gradients over multiple mini-batches to simulate a larger effective batch size (per_device_train_batch_size * gradient_accumulation_steps). Default is 1.
    warmup_steps = 20,                # Warmup phase: The number of steps for the learning rate to linearly increase from 0 to its initial value. Default is 0.
    max_steps = 3500,                 # Total training steps: The total number of update steps to perform. This overrides num_train_epochs. Default is -1 (no limit).
    learning_rate = 1e-4,             # Learning rate: The initial learning rate for the optimizer. Default is 5e-5.
    fp16 = not torch.cuda.is_bf16_supported(), # Mixed precision training: Uses 16-bit floating point numbers for training to save memory and speed up computation. Automatically set based on BF16 support. Default is False.
    bf16 = torch.cuda.is_bf16_supported(),     # Bfloat16 training: Uses bfloat16 for training, typically available on Ampere+ GPUs. Automatically set based on BF16 support. Default is False.
    logging_steps = 1,                # Logging frequency: The number of update steps between two loggings. Default is 500.
    optim = 'adamw_8bit',             # Optimizer: Specifies the optimizer to use. 'adamw_8bit' is selected for memory efficiency. Default is 'adamw_torch'.
    weight_decay = 0.01,              # Weight decay: The strength of L2 regularization. Default is 0.
    lr_scheduler_type = 'linear',     # Learning rate scheduler: Defines how the learning rate changes over time. 'linear' decays linearly after warmup. Default is 'linear'.
    seed = 3407,                      # Random seed: Sets the random seed for reproducibility. Default is None.
    output_dir = 'outputs',           # Output directory: The directory where model checkpoints and predictions will be saved. Default is './'.
    eval_strategy = 'steps',          # Evaluation strategy: Defines when evaluation is performed. 'steps' evaluates every eval_steps. Default is 'no'.
    eval_steps = 350,                 # Evaluation frequency: The number of update steps between two evaluations when eval_strategy is 'steps'. Default is 500.
    save_strategy = 'steps',          # Save strategy: Defines when checkpointing is performed. 'steps' saves every save_steps. Default is 'steps'.
    save_steps = 350,                 # Save frequency: The number of update steps between two checkpoint savings when save_strategy is 'steps'. Default is 500.
    load_best_model_at_end = True,    # Load best model: Whether to load the best model found during training at the end of training. Default is False.
    metric_for_best_model = 'loss',   # Best model metric: The metric to use to compare models during evaluation (e.g., 'loss', 'accuracy'). Default is 'eval_loss'.
    report_to = 'none',               # Reporting tools: Disables integration with external reporting tools like Weights & Biases or TensorBoard. Default is 'all'.
)

# b. Set up early stopping and checkpoint
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    args = training_args,
    # Early stopping stops training if the validation loss doesn't improve for 3 evaluations
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# c. Execute fine-tuning
trainer.train()

# d. Save fine-tuned model
# This saves the LoRA adapters and the tokenizer.
model.save_pretrained('lora_model_final')
tokenizer.save_pretrained('lora_model_final')
print('Model saved successfully.')

## 4. Evaluation and Analysis

In [ ]:
# a. Compare pre and post fine-tuning performance
# b. Analyse the outputs (3 examples)

# Switch model to inference mode
FastLanguageModel.for_inference(model)

test_queries = [
    'If x + 5 = 10, what is x?',
    'What is the derivative of x^2?',
    'A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?'
]

for query in test_queries:
    # Prepare input using the same prompt template used in training
    inputs = tokenizer([alpaca_prompt.format(query, '')], return_tensors='pt').to('cuda')

    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    response = tokenizer.batch_decode(outputs)[0]

    print(f'Query: {query}')
    # We strip the prompt part to show only the generated response
    print(f'Generated Response:\n{response.split('### Response:')[1]}')
    print('-' * 30)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: If x + 5 = 10, what is x?
Generated Response:

We can solve this equation by isolating the variable x.
We can start by subtracting 5 from both sides of the equation:
x + 5 - 5 = 10 - 5
x = 5
Therefore, the value of x is 5.
The answer is: 5<|end_of_text|>
------------------------------


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What is the derivative of x^2?
Generated Response:

The derivative of $x^2$ is $2x$.
So, the answer is $\boxed{2x}$.The answer is: 2x<|end_of_text|>
------------------------------
Query: A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?
Generated Response:

If the train travels 60 miles in 1 hour, then it travels 60 miles/hour * 1 hour = 60 miles in 1 hour.
If it travels 2.5 hours, then it travels 2.5 hours * 60 miles/hour = 150 miles.
#### 150
The answer is: 150<|end_of_text|>
------------------------------


### 4c. Discuss challenges and limitations
1. **Hardware Constraints**: The free tier of Google Colab (T4 GPU) has limited VRAM (16GB), which necessitates the use of 4-bit quantization and small batch sizes.
2. **Dataset Size**: Used a 10,000-sample subset for efficiency. A full fine-tuning on all 395k samples would yield better results but requires significantly more time and compute.
3. **Model Size**: Llama 3.2 1B is a very small model. While surprisingly capable, it may hallucinate on extremely complex mathematical proofs that require deep symbolic reasoning.
4. **Performance**:  

**4.a** First test run with `test_queries` returned obviously erroneous answers, all 3! Independently checked, twice.  
**4.b** `max_steps` was set to 60, which is a very short training duration for a task like mathematical reasoning. This likely explains the poor performance.  
**4.c** `eval_steps` was set to 20, might need more frequent evaluation during a longer training period.
The inaccurate answers from `test_queries` points to a longer training period is needed to give the model more opportunities to learn from the dataset.  
**4.d** modify the `max_steps` to `200` and `eval_steps` to `50` in the `TrainingArguments` to allow for more training and evaluation.  
**4.e** Then Part 3 and 4 were re-run to see if the inference improves.  
**4.f** After increasing `max_steps` and `eval_steps` it did give correct answers to all 3 questions in the `test_queries`.  
**4.g** BUT further testing, just before finalising and uploading, still shows some persistent errors in derivative and train reasoning problems.  
**4.h** Decided to re-train with increasing `step` values until the `test_queries` returns all correct answers.  

Ran an AI analysis on the training configuration, previous attempts at increasing `steps` did not yield improvement :
With max_steps is currently set to 600. With 9,000 training examples and an effective batch size of 8 (2 * 4), one full epoch of training data would require 1,125 steps (9,000 / 8). This means the model hasn't even seen all the training data once yet.  

To address this, further increase the max_steps from 600 to 2000. This will allow the model to go through the dataset roughly 1.7 times, giving it more opportunities to learn and refine its reasoning.

Also adjust eval_steps and save_steps to 200 to keep evaluation and checkpointing consistent with the longer training duration.  

**Result of this change:** Progress made on the algebraic and derivative problem. BUT, in one evaluation instance, the response to the `train` word problem was wrong because it reasoned a need to convert hours to minutes and multiply by miles/hour. Very strange.  

**Action:** further increase the max_steps for fine-tuning from 2000 to 3500. This will allow the model to train for roughly 3 full epochs, giving it more opportunities to learn the patterns necessary for solving word problems. Also adjust eval_steps and save_steps to 350 to match the extended training duration.


**Summary of this adjustment:** Observed a trade-off. While the train problem has improved significantly, the model has regressed on the simpler algebraic and derivative problems.

This often indicates that with more training steps at the current learning rate, the model might be overfitting to some patterns or forgetting others.

Since the max_steps of 3500 (4 epochs) allowed the model to correctly solve the more complex train problem, retain that increased training duration.

To address the regression on the other problems and encourage more stable learning, lower the learning_rate from 2e-4 to **1e-4** (AI recommendation).

This will allow the model to adjust its weights more gradually during the 3500 training steps, hopefully helping it to retain the correct solutions for the algebraic and derivative problems while maintaining the improved performance on the train problem.

**4.i** The model is still struggling to correctly solve all three types of problems simultaneously, even after previous adjustments.

Currently, the model is getting the algebraic and derivative problems correct, but the word problem about the train is incorrect.

(AI Analysis) This behavior often indicates that the model is either overfitting to certain patterns or struggling to generalize across the different problem types within the dataset.

**Action:** (AI recommendations)

Make two changes to `TrainingArguments`  
Increase `gradient_accumulation_steps` from 4 to 8: This will effectively double batch size from 8 to 16 (per_device_train_batch_size * gradient_accumulation_steps), allowing the model to process more examples before making an update.  

A larger effective batch size can lead to more stable gradients and better generalization, as the model sees a broader representation of the data in each update step.

Increase `warmup_steps` from 10 to 20:

A slightly longer warmup period can help the learning rate schedule stabilize at the beginning of training, which can sometimes lead to better overall convergence for complex tasks.

These changes aim to help the model generalize better across the diverse problem types present in the `MetaMathQA` dataset.

**4.j It gave the correct answer to all 3 queries this time.**

The standard AI disclaimer "AI can make mistakes. Check the output" really does apply in Fine-Tuning this model.

It responded with all 3 correct answers in a test runs but the first try had an error on the train word problem. It did that on the second try after first adjustment.

And the later problems gleaned from testing seems to be that it can cope with algebraic and derivative problems but not at the same time as word problems which required language reasoning.

Getting correct answers on all 3 types of test questions after fine-tuning is a very positive sign, but it does not automatically guarantee that the model is fully 'stable' or will give the correct answer every single time on new, unseen problems.

All this leads to the question: When and how does the fine-tuned model becomes stable and will give the correct answer each time?

(AI Recommendations)

**Why consistency on test_queries doesn't guarantee full stability:**

**Limited Test Set:** The `test_queries` are a very small sample (just 3 examples). The model might have, through fine-tuning, 'memorized' these specific examples or learned patterns that happen to work for them, but these patterns might not generalize to a broader range of mathematical problems.

**Overfitting (still a possibility):** Even with the gradient_accumulation_steps and warmup_steps adjustments, there's always a risk of overfitting. A model that overfits performs very well on the training and validation data (and often small test sets it might have indirectly learned from) but poorly on truly novel data.

**Stochastic Nature:** Large language models, even after fine-tuning, can exhibit some degree of stochasticity in their responses, especially for more complex problems or when the input is slightly ambiguous. This means generating the same response every time for an identical input can sometimes depend on factors like temperature settings during inference.


**How to ensure more robust stability and consistent correct answers:**

**Expanded and Diverse Evaluation Set:**

**Larger Test Set:** Crucially, a much larger and more diverse test set is needed; one that the model has never seen during training or validation. This set should cover a wide range of mathematical problem types and difficulties present in your target domain. The 1000 validation samples (val_dataset) is a good starting point, but even more dedicated, unseen test data would be better. Evaluate the model's performance on this larger set using appropriate metrics (e.g., accuracy, exact match).

**Edge Cases and Tricky Problems:** Include problems that test the model's understanding of specific mathematical concepts, potential ambiguities, or common pitfalls.

**Quantitative Metrics and Error Analysis:**

Beyond just checking individual answers, calculate quantitative metrics like accuracy or exact match score on your evaluation datasets. This gives you a statistical measure of how often the model is correct.

**Perform error analysis:** For the problems the model gets wrong, analyze why it made the mistake. Was it a misinterpretation of the query? A calculation error? A logical fallacy? This can reveal systematic weaknesses.

**Hyperparameter Tuning (Continuous Refinement):**

The adjustments made were good steps; it improved the answers. Further fine-tuning of `learning_rate`, `warmup_steps`, `gradient_accumulation_steps`, and `max_steps` can continue to improve generalization.

May consider using more advanced tuning techniques if needed.
Could experiment with `lora_alpha` and `r` (LoRA rank); if the model still seems to be struggling to learn complex patterns.

**Ensembling (Advanced):** For extremely critical applications, it is suggested to run multiple fine-tuned models (perhaps trained with slightly different hyperparameters or on different data splits) and combine their predictions. If multiple models agree, the results have higher confidence.

**Beam Search / Diverse Beam Search (During Inference):** Instead of just `model.generate()`, explore more sophisticated decoding strategies like beam search or diverse beam search. These methods explore multiple possible output sequences, which can sometimes lead to finding a correct answer that a simple greedy decoding might miss.

**In summary,** while the current results are encouraging, true stability comes from rigorous, broad evaluation against unseen data and continuous refinement of the training process.


**Before adjustment (first)**:  
```
1. Generated Response:
We can write x + 5 = 10 as x = 5 + 10 - 5 = 10.
Therefore, x = 10.  
The answer is: 10<|end_of_text|>  

2. Generated Response:  
To find the derivative of x^2,  
we need to find the derivative of x.  
We can use the quotient rule to find the derivative of x.  
The quotient rule states that the derivative of x is  
x times the derivative of x minus 1.  
So, the derivative of x^2 is x times the derivative of x minus 1.  
To find the derivative of x, we can use the power rule.  
The power rule states that the derivative of x^k is k times the derivative of x.  
So, the derivative of x^2 is 2x times the derivative of x minus 1. Therefore, the derivative  
------------------------------  
(seems incomplete)

3. Query: A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?  
Generated Response:  
In 1 hour, the train travels 60 miles.  
In 2.5 hours, the train travels 60/2.5 = 24 miles.  
Therefore, the train travels 24 miles in 2.5 hours.  
#### 24  
The answer is: 24<|end_of_text|>  
```

**After first adjustment (second test)**:  
```
Query: If x + 5 = 10, what is x?  

Generated Response:  
We are given that $x + 5 = 10$.
Simplifying, we have $x + 5 = 10$.
Subtracting 5 from both sides, we get $x = 5$.
Therefore, x + 5 = 10, so x = 5.
The answer is: 5<|end_of_text|>
------------------------------

Query: What is the derivative of x^2?

Generated Response:
We can differentiate x^2 using the product rule.  
The derivative of x^2 is 2x.  
The derivative of x^2 is 2x.<|end_of_text|>  
------------------------------  

Query: A train travels 60 miles in 1 hour. How far does it travel in 2.5 hours?  

Generated Response:
To find the distance traveled by the train in 2.5 hours, we need to calculate the distance traveled in 60 miles divided by 60 miles/hour.
60 miles divided by 60 miles/hour = 1 hour
Therefore, the train travels 60 miles in 1 hour.
To find the distance traveled in 2.5 hours, we can multiply the distance traveled in 1 hour by 2.5 hours.
60 miles * 2.5 hours = 150 miles
Therefore, the train travels 150 miles in 2.5 hours.
#### 150
The answer is: 150
------------------------------
```
Not sure why the 2nd evaluation gave correct all 3 answers but subsequently not. Could by overfitting and memorising examples or just luck, just happened to pick the right logic to the right answer?

## 5. Documentation and Report
### Implementation Documentation
The code is documented with inline comments explaining parameters such as `learning_rate`, `batch_size`, and `r` (LoRA rank).  
Default values like `lora_dropout=0` are used as per Unsloth's optimizations.  

Actual parameter values may have changed due to re-training.

Expanded verbose comments for clarity and as a self-reminder.
```
# --- Load Model ---
# We use the pre-quantized 4-bit version of Llama 3.2 1B from Unsloth.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-1B-bnb-4bit', # The name or path of the
                                                  # pre-trained model to load. # No default,
                                                  # must be specified.
    max_seq_length = max_seq_length,              # Maximum sequence length
                                                  # the model can handle.
                                                  # Default varies by model,
                                                  # often 2048 or 4096.
    dtype = dtype,                                # Data type for model
                                                  # weights (e.g., torch.
                                                  # float16, torch.bfloat16).
                                                  # 'None'
                                                  # auto-detects based on GPU.
                                                  # Default is None.
    load_in_4bit = load_in_4bit,                  # Whether to load
                                                  # the model in 4-bit
                                                  # quantization, saving VRAM.
                                                  # Default is False.
)

# --- Configure LoRA Adapters ---
# LoRA allows us to train only a small fraction (1-10%)
# of the model parameters.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA attention dimension (rank):
                          # Controls the number of trainable parameters
                          # in LoRA.
                          # Higher values allow more complex updates
                          # but use more VRAM.
                          # Default is 64.
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                          # List of module names to apply LoRA to.
                          # Default depends on the model architecture.
    lora_alpha = 16,      # LoRA scaling factor:
                          # Multiplies the LoRA weights.
                          # Default is 16.
    lora_dropout = 0,     # The dropout probability for LoRA layers.
                          # Optimized to 0 for Unsloth's performance.
                          # Default is 0.05.
    bias = 'none',        # Whether to train the bias terms in LoRA layers.
                          # Optimized to 'none' for Unsloth.
                          # Default is 'none'.
    use_gradient_checkpointing = 'unsloth', # Reduces VRAM usage by
                          # recomputing activations during backpropagation.
                          # 'unsloth' uses an optimized version.
                          # Default is True.
    random_state = 3407,  # Random seed for reproducibility of
                          # LoRA layer initialization.
                          # Default is None.
    use_rslora = False,   # Whether to use Rank Stabilized LoRA.
                          # Default is False.
    loftq_config = None,  # Configuration for LoftQ initialization.
                          # Default is None.
)
```
**Adding more verbose comments for clarification but parameter values have changed**
```
# a. Configure hyperparameters  
training_args = TrainingArguments(  
    per_device_train_batch_size = 2,  # Batch size per GPU:
                                      # This sets the number of training
                                      # examples processed per device in a
                                      # single forward/backward pass.
                                      # Default is 8.  
    gradient_accumulation_steps = 8,  # Accumulate gradients:
                                      # This accumulates gradients over
                                      # multiple mini-batches to simulate a
                                      # larger effective batch size
                                      # (per_device_train_batch_size *
                                      # gradient_accumulation_steps).
                                      # Default is 1.  
    warmup_steps = 20,                # Warmup phase: The number of steps
                                      # the learning rate to linearly increase
                                      # from 0 to its initial value.
                                      # Default is 0.  
    max_steps = 3500,                 # Total training steps: The total number
                                      # of update steps to perform.
                                      # This overrides num_train_epochs.
                                      # Default is -1 (no limit).  
    learning_rate = 1e-4,             # Learning rate: The initial learning
                                      # rate for the optimizer.
                                      # Default is 5e-5.  
    fp16 = not torch.cuda.is_bf16_supported(), # Mixed precision training:
                                               # Uses 16-bit floating point
                                               # numbers for training to save
                                               # memory and speed up
                                               # computation.
                                               # Automatically set based on
                                               # BF16 support.
                                               # Default is False.  
    bf16 = torch.cuda.is_bf16_supported(),     # Bfloat16 training: Uses
                                               # bfloat16 for training,
                                               # typically available on
                                               # Ampere GPUs.
                                               # Automatically set based
                                               # on BF16 support.
                                               # Default is False.  
    logging_steps = 1,                # Logging frequency: The number of
                                      # update steps between two loggings.
                                      # Default is 500.  
    optim = 'adamw_8bit',             # Optimizer: Specifies the optimizer
                                      # to use.
                                      # 'adamw_8bit' is selected for memory # efficiency.
                                      # Default is 'adamw_torch'.  
    weight_decay = 0.01,              # Weight decay: The strength of
                                      # L2 regularization.
                                      # Default is 0.  
    lr_scheduler_type = 'linear',     # Learning rate scheduler: Defines how
                                      # the learning rate changes over time.
                                      # 'linear' decays linearly after warmup.
                                      # Default is 'linear'.  
    seed = 3407,                      # Random seed: Sets the random seed
                                      # for reproducibility.
                                      # Default is None.  
    output_dir = 'outputs',           # Output directory: The directory where
                                      # model checkpoints and predictions
                                      # will be saved.
                                      # Default is './'.  
    eval_strategy = 'steps',          # Evaluation strategy: Defines when
                                      # evaluation is performed.
                                      # 'steps' evaluates every eval_steps.
                                      # Default is 'no'.  
    eval_steps = 350,                 # Evaluation frequency: The number of
                                      # update steps between two evaluations
                                      # when eval_strategy is 'steps'.
                                      # Default is 500.  
    save_strategy = 'steps',          # Save strategy: Defines when
                                      # checkpointing is performed.
                                      # 'steps' saves every save_steps.
                                      # Default is 'steps'.  
    save_steps = 350,                 # Save frequency: The number of update
                                      # steps between two checkpoint savings
                                      # when save_strategy is 'steps'.
                                      # Default is 500.  
    load_best_model_at_end = True,    # Load best model: Whether to load
                                      # the best model found during training
                                      # at the end of training.
                                      # Default is False.  
    metric_for_best_model = 'loss',   # Best model metric: The metric to use
                                      # to compare models during evaluation
                                      # (e.g., 'loss', 'accuracy').
                                      # Default is 'eval_loss'.  
    report_to = 'none',               # Reporting tools: Disables integration
                                      # with external reporting tools like
                                      # Weights & Biases or TensorBoard.
                                      # Default is 'all'.  
```
### Training Logs and Metrics
Training logs (loss, learning rate, and step time) are automatically printed by the `SFTTrainer` during the `trainer.train()` execution.   
Final parameter settings  
`max_steps` of **3500 (4 epochs)** allowed the model to correctly solve the more complex train problem  
`learning_rate` from 2e-4 to **1e-4**  
adjusted `eval_steps` and `save_steps` to **350** to keep evaluation and checkpointing consistent with the longer training duration.
```
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 7 | Total steps = 3,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
 [3150/3500 2:39:25 < 17:43, 0.33 it/s, Epoch 5/7]
Step	Training Loss	Validation Loss
350	0.698248	0.677785
700	0.665121	0.649980
1050	0.601167	0.631010
1400	0.562012	0.621439
1750	0.445872	0.621691
2100	0.424265	0.613663
2450	0.458348	0.621121
2800	0.364262	0.616981
3150	0.407444	0.625882

```

Training was limited to the free tier of Google Colab; using 2 gmail accounts. Gemini AI side panel assist was turn on.
